# 02 · Transient 1D Heat PINN

Extends the day-1 steady-state skeleton to
∂T/∂t = α ∂²T/∂x² on x ∈ [0,1], t ∈ [0, 0.2] with T(0,t)=T(1,t)=0
and T(x,0)=sin(πx). Analytical: T = sin(πx) exp(-α π² t).

In [8]:
%load_ext autoreload
%autoreload 2

# --- Path shim: make the repo root importable from notebooks/ ---
import sys, pathlib
_repo_root = pathlib.Path.cwd().parent   # notebooks/ → project root
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
# ----------------------------------------------------------------

import math
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

from src.train import (
    train_transient, analytical_solution, l2_error, plot_loss_history,
)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
ALPHA   = 1.0
T_FINAL = 0.2

model, history = train_transient(
    alpha=ALPHA, t_final=T_FINAL,
    n_pde=4000, n_bc=200, n_ic=200,
    epochs=5000, lr=1e-3,
    w_pde=1.0, w_bc=10.0, w_ic=10.0,
    log_every=500, seed=0,
)

ep     0 | L=4.470e+00 (pde 8.01e-02, bc 6.81e-02, ic 3.71e-01)
ep   500 | L=8.309e-01 (pde 6.48e-02, bc 4.92e-02, ic 2.74e-02)
ep  1000 | L=9.348e-02 (pde 4.07e-02, bc 3.40e-03, ic 1.88e-03)
ep  1500 | L=3.018e-02 (pde 1.44e-02, bc 1.12e-03, ic 4.53e-04)
ep  2000 | L=1.116e-02 (pde 6.63e-03, bc 3.67e-04, ic 8.59e-05)
ep  2500 | L=5.888e-03 (pde 3.19e-03, bc 1.96e-04, ic 7.35e-05)
ep  3000 | L=2.403e-03 (pde 2.15e-03, bc 1.43e-05, ic 1.07e-05)
ep  3500 | L=1.660e-02 (pde 2.07e-03, bc 7.49e-04, ic 7.04e-04)
ep  4000 | L=5.435e-03 (pde 1.50e-03, bc 1.83e-04, ic 2.10e-04)
ep  4500 | L=1.024e-03 (pde 9.03e-04, bc 5.39e-06, ic 6.72e-06)
ep  4999 | L=1.271e-02 (pde 1.12e-03, bc 5.67e-04, ic 5.92e-04)


In [10]:
plot_loss_history(history)
plt.title("Composite loss (log scale)")
plt.show()

In [11]:
err = l2_error(model, alpha=ALPHA, t_final=T_FINAL)
print(f"L2 error over (x,t) strip: {err:.3e}")

L2 error over (x,t) strip: 1.807e-02


In [12]:
nx = 200
x  = torch.linspace(0, 1, nx).unsqueeze(1)
fig, ax = plt.subplots(figsize=(7, 4))
for t_val in [0.00, 0.05, 0.20]:
    t = torch.full_like(x, t_val)
    with torch.no_grad():
        T_pred = model(torch.cat([x, t], 1)).squeeze()
    T_true = analytical_solution(x, t, alpha=ALPHA).squeeze()
    ax.plot(x.squeeze(), T_true,  "--", label=f"analytical t={t_val}")
    ax.plot(x.squeeze(), T_pred,  "-",  label=f"PINN t={t_val}")
ax.set_xlabel("x"); ax.set_ylabel("T"); ax.legend(); ax.grid(alpha=0.3)
plt.show()

In [13]:
nx, nt = 200, 60
x_np = np.linspace(0, 1, nx)
t_np = np.linspace(0, T_FINAL, nt)

x = torch.tensor(x_np, dtype=torch.float32).unsqueeze(1)
frames_pred, frames_true = [], []
for tv in t_np:
    t = torch.full_like(x, float(tv))
    with torch.no_grad():
        frames_pred.append(model(torch.cat([x, t], 1)).squeeze().numpy())
    frames_true.append(np.sin(np.pi * x_np) * np.exp(-ALPHA * np.pi**2 * tv))

fig, ax = plt.subplots(figsize=(7, 4))
line_true, = ax.plot([], [], "--", label="analytical")
line_pred, = ax.plot([], [], "-",  label="PINN")
ax.set_xlim(0, 1)
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel("x")
ax.set_ylabel("T")
ax.legend()
ax.grid(alpha=0.3)
title = ax.set_title("")

def init():
    line_true.set_data([], [])
    line_pred.set_data([], [])
    return line_true, line_pred, title

def update(i):
    line_true.set_data(x_np, frames_true[i])
    line_pred.set_data(x_np, frames_pred[i])
    t_val = float(t_np[i])
    title.set_text("t = %.3f" % t_val)
    return line_true, line_pred, title

anim = animation.FuncAnimation(fig, update, frames=nt, init_func=init, blit=True, interval=60)
HTML(anim.to_jshtml())

In [14]:
# Diagnose causality: residual magnitude as function of t
with torch.no_grad():
    pass  # placeholder — full check below uses grad, so no torch.no_grad
x_g = torch.rand(2000, 1)
t_g = torch.rand(2000, 1) * T_FINAL
from src.pde_loss import transient_residual
r = transient_residual(model, x_g, t_g, alpha=ALPHA).detach().abs()

bins = np.linspace(0, T_FINAL, 11)
idx = np.digitize(t_g.numpy().flatten(), bins) - 1
mean_r = [r.numpy().flatten()[idx == k].mean() for k in range(len(bins) - 1)]
plt.plot(0.5 * (bins[:-1] + bins[1:]), mean_r, "o-")
plt.xlabel("t"); plt.ylabel("|residual|"); plt.title("Residual vs t")
plt.grid(alpha=0.3); plt.show()